# MicroFinML - Big Data Analytics Analysis
## Distributed Processing, Scalability, Blockchain Audit & Social Impact

This notebook demonstrates:
1. **Spark distributed preprocessing** — Loading and transforming data at scale
2. **Spark MLlib model training** — Distributed ML vs local scikit-learn
3. **Scalability analysis** — Execution time vs dataset size benchmarks
4. **Blockchain audit trail** — Immutable ledger for credit decisions
5. **Social impact analysis** — Economic inclusion and sustainability metrics
6. **5Vs of Big Data** — Volume, Velocity, Variety, Veracity, Value in microfinance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os, time, json, warnings

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded.')

---
## Part 1: The 5Vs of Big Data in Microfinance Context

### Volume
- Our dataset contains **255,347 loan records** — representative of real MFI transaction logs
- Real microfinance systems process millions of transactions daily across global networks
- Requires distributed storage (HDFS) and processing (Spark) for production scale

### Velocity
- Loan applications arrive in real-time; credit decisions must be made within seconds
- Stream processing (Kafka → Spark Streaming) enables real-time risk scoring
- Our batch pipeline demonstrates the core ML logic; production would add streaming layer

### Variety
- **Structured data:** Numeric (income, credit score) + Categorical (education, employment)
- **Semi-structured:** JSON loan applications, XML regulatory filings
- **Unstructured:** Borrower notes, community feedback, alternative data sources
- Our dataset has 9 numeric + 7 categorical features = mixed variety

### Veracity
- Credit data may contain errors, missing fields, or fraudulent information
- Our dataset is clean (0 missing values), but real MFI data requires robust validation
- Data quality pipelines (Spark-based ETL) are essential for trustworthy predictions

### Value
- Accurate default prediction → reduced financial losses for MFIs
- Better credit scoring → financial inclusion for underserved populations
- Data-driven decisions → sustainable economic growth in micro-financial ecosystems

---
## Part 2: Spark Distributed Data Processing

In [ ]:
from src.spark_processing import (
    create_spark_session, load_data_spark,
    spark_feature_engineering, spark_data_profiling,
    spark_preprocess_pipeline
)

spark = create_spark_session(app_name='MicroFinML-BDA')

In [ ]:
# Run full Spark preprocessing pipeline
spark_data = spark_preprocess_pipeline(spark, '../data/raw/Loan Default.csv')

In [ ]:
# Show Spark execution plan (demonstrates distributed query planning)
print('=== Spark Execution Plan ===')
print('This shows how Spark optimizes and distributes the computation:')
spark_data['train_df'].select('features', 'Default').explain(True)

---
## Part 3: Spark MLlib — Distributed Model Training

In [ ]:
from src.spark_ml_training import train_and_evaluate_all_spark, generate_spark_comparison_table

# Select features and label for ML
train_ml = spark_data['train_df'].select('features', 'Default')
test_ml = spark_data['test_df'].select('features', 'Default')

print(f'Training set: {train_ml.count():,} samples')
print(f'Test set: {test_ml.count():,} samples')

In [ ]:
# Train all Spark MLlib models
spark_results = train_and_evaluate_all_spark(
    train_ml, test_ml,
    save_dir='../models/spark'
)

In [ ]:
# Generate comparison: Local scikit-learn vs Spark MLlib
comparison = generate_spark_comparison_table(
    spark_results,
    local_metrics_path='../results/metrics/model_performance.csv'
)

print('\n=== Framework Comparison: scikit-learn (Local) vs Spark MLlib ===')
comparison

---
## Part 4: Scalability Analysis

In [ ]:
from src.scalability_analysis import run_full_scalability_analysis

scalability_results = run_full_scalability_analysis(
    data_path='../data/raw/Loan Default.csv',
    spark=spark,
    save_dir='../results'
)

In [ ]:
# Display scalability results table
print('\n=== Scalability Results ===')
scalability_results.round(4)

In [ ]:
# Show generated scalability plots
from IPython.display import Image, display
fig_dir = '../results/figures/model_comparison'
for fname in ['scalability_local.png', 'scalability_spark.png', 'scalability_comparison.png']:
    path = os.path.join(fig_dir, fname)
    if os.path.exists(path):
        print(f'\n--- {fname} ---')
        display(Image(filename=path))

---
## Part 5: Blockchain Audit Trail for Credit Decisions

In [ ]:
from src.blockchain_audit import LoanAuditChain
import joblib
from src.prediction import predict_single

# Initialize blockchain
audit_chain = LoanAuditChain()

# Load the best model and preprocessor
best_model = joblib.load('../models/xgboost_model.pkl')
preprocessor = joblib.load('../data/processed/preprocessor.pkl')

# Sample loan applications to score and record
test_applications = [
    {'Age': 35, 'Income': 55000, 'LoanAmount': 25000, 'CreditScore': 680,
     'MonthsEmployed': 48, 'NumCreditLines': 3, 'InterestRate': 12.5,
     'LoanTerm': 36, 'DTIRatio': 0.35, 'Education': "Bachelor's",
     'EmploymentType': 'Full-time', 'MaritalStatus': 'Married',
     'HasMortgage': 'Yes', 'HasDependents': 'Yes',
     'LoanPurpose': 'Home', 'HasCoSigner': 'No'},
    {'Age': 22, 'Income': 18000, 'LoanAmount': 45000, 'CreditScore': 420,
     'MonthsEmployed': 6, 'NumCreditLines': 1, 'InterestRate': 22.0,
     'LoanTerm': 60, 'DTIRatio': 0.85, 'Education': 'High School',
     'EmploymentType': 'Part-time', 'MaritalStatus': 'Single',
     'HasMortgage': 'No', 'HasDependents': 'No',
     'LoanPurpose': 'Auto', 'HasCoSigner': 'No'},
    {'Age': 50, 'Income': 95000, 'LoanAmount': 30000, 'CreditScore': 750,
     'MonthsEmployed': 96, 'NumCreditLines': 4, 'InterestRate': 8.5,
     'LoanTerm': 24, 'DTIRatio': 0.2, 'Education': "Master's",
     'EmploymentType': 'Full-time', 'MaritalStatus': 'Married',
     'HasMortgage': 'Yes', 'HasDependents': 'Yes',
     'LoanPurpose': 'Education', 'HasCoSigner': 'Yes'},
    {'Age': 28, 'Income': 32000, 'LoanAmount': 80000, 'CreditScore': 380,
     'MonthsEmployed': 12, 'NumCreditLines': 2, 'InterestRate': 24.0,
     'LoanTerm': 48, 'DTIRatio': 0.78, 'Education': 'High School',
     'EmploymentType': 'Unemployed', 'MaritalStatus': 'Divorced',
     'HasMortgage': 'No', 'HasDependents': 'Yes',
     'LoanPurpose': 'Business', 'HasCoSigner': 'No'},
    {'Age': 42, 'Income': 75000, 'LoanAmount': 20000, 'CreditScore': 700,
     'MonthsEmployed': 60, 'NumCreditLines': 3, 'InterestRate': 10.0,
     'LoanTerm': 36, 'DTIRatio': 0.3, 'Education': "Bachelor's",
     'EmploymentType': 'Self-employed', 'MaritalStatus': 'Single',
     'HasMortgage': 'Yes', 'HasDependents': 'No',
     'LoanPurpose': 'Other', 'HasCoSigner': 'Yes'}
]

print('Recording loan decisions to blockchain audit trail...\n')
for app in test_applications:
    result = predict_single(app, best_model, preprocessor)
    # Record to blockchain
    loan_summary = {k: v for k, v in app.items() if k in ['Age', 'Income', 'LoanAmount', 'CreditScore', 'LoanPurpose']}
    block = audit_chain.add_decision(
        loan_data=loan_summary,
        prediction=result['prediction_label'],
        risk_score=result['default_probability'],
        model_used='XGBoost'
    )
    print(f"  Block #{block.index}: {result['prediction_label']} "
          f"(risk: {result['default_probability']:.2%}) "
          f"[{block.hash[:24]}...]")

In [ ]:
# Validate blockchain integrity
is_valid, message = audit_chain.validate_chain()
print(f'\nBlockchain Validation: {"PASSED" if is_valid else "FAILED"}')
print(f'  {message}')

# Print chain
audit_chain.print_chain()

# Summary
summary = audit_chain.get_chain_summary()
print(f'\nAudit Chain Summary:')
print(f'  Total decisions:  {summary["total_decisions"]}')
print(f'  Predicted Repay:  {summary["repay_count"]}')
print(f'  Predicted Default: {summary["default_count"]}')
print(f'  Default rate:     {summary["default_rate"]:.2%}')

In [ ]:
# Show blockchain as DataFrame
chain_df = audit_chain.to_dataframe()
print('\n=== Blockchain Ledger (DataFrame View) ===')
chain_df

In [ ]:
# Tamper detection demonstration
print('=== Tamper Detection Demo ===')
print('\nOriginal chain validation:', audit_chain.validate_chain()[1])

# Attempt to tamper
print('\nAttempting to change Block #2 prediction from DEFAULT to REPAY...')
original_pred = audit_chain.chain[2].prediction
audit_chain.chain[2].prediction = 'REPAY'  # Tamper!
is_valid, msg = audit_chain.validate_chain()
print(f'Validation result: {"PASSED" if is_valid else "FAILED"}')
print(f'  {msg}')

# Restore
audit_chain.chain[2].prediction = original_pred
audit_chain.chain[2].hash = audit_chain.chain[2].compute_hash()
print('\nBlock restored. Chain re-validated:', audit_chain.validate_chain()[1])

---
## Part 6: Social Impact Analysis

### Economic Impact of ML-Based Credit Scoring in Microfinance

In [ ]:
# Load test results to quantify social impact
metrics = pd.read_csv('../results/metrics/model_performance.csv', index_col=0)

# Use the best model's metrics for impact analysis
best_model_name = metrics['roc_auc'].idxmax()
recall = metrics.loc[best_model_name, 'recall']
precision = metrics.loc[best_model_name, 'precision']
accuracy = metrics.loc[best_model_name, 'accuracy']

# Social impact calculations
total_loans = 255347
default_rate = 0.1161
avg_loan_amount = 127578  # from dataset stats
total_defaults = int(total_loans * default_rate)
total_default_value = total_defaults * avg_loan_amount

# With ML-based scoring
defaults_caught = int(total_defaults * recall)
money_saved = defaults_caught * avg_loan_amount
false_alarms = int(defaults_caught / precision - defaults_caught) if precision > 0 else 0
good_loans_denied = false_alarms

print('=' * 65)
print('  SOCIAL IMPACT ANALYSIS: ML-Based Credit Scoring')
print('  MicroFinML for Sustainable Economics')
print('=' * 65)

print(f'\n  Best Model: {best_model_name}')
print(f'  Dataset: {total_loans:,} loan applications')
print(f'  Average loan amount: ${avg_loan_amount:,}')
print(f'  Historical default rate: {default_rate:.2%}')

print(f'\n  --- Without ML Scoring ---')
print(f'  Total defaults: {total_defaults:,}')
print(f'  Total loss from defaults: ${total_default_value:,.0f}')

print(f'\n  --- With ML Scoring (MicroFinML) ---')
print(f'  Defaults identified early: {defaults_caught:,} ({recall:.1%} recall)')
print(f'  Estimated money saved: ${money_saved:,.0f}')
print(f'  False alarms (good loans flagged): {good_loans_denied:,}')

print(f'\n  --- Economic Sustainability Impact ---')
reduction_pct = (defaults_caught / total_defaults) * 100 if total_defaults > 0 else 0
print(f'  Default risk reduction: {reduction_pct:.1f}%')
print(f'  More borrowers can access credit with confidence')
print(f'  MFIs can sustain operations and expand lending')
print(f'  Promotes financial inclusion for underserved populations')
print('=' * 65)

---
## Part 7: Ethical Implications & Data Privacy Discussion

### Ethical Considerations in ML-Based Credit Scoring

**1. Algorithmic Bias**
- ML models may inadvertently discriminate based on demographic features (Age, Education, MaritalStatus)
- Features like EmploymentType could disadvantage self-employed or gig workers
- Fairness metrics should be computed across protected groups
- Regular bias audits are essential before deployment

**2. Data Privacy & Security**
- Financial data is highly sensitive (income, credit score, debt ratios)
- Compliance with regulations (GDPR, India's DPDP Act) is mandatory
- Data must be encrypted at rest and in transit
- Access controls and audit trails (blockchain) ensure accountability

**3. Veracity of Input Data**
- Noisy or fraudulent applications can lead to incorrect predictions
- Data validation pipelines (Spark-based ETL) help filter anomalies
- Alternative data sources should be carefully vetted for reliability

**4. Transparency & Explainability**
- Borrowers have a right to know why their loan was denied
- Feature importance analysis provides model interpretability
- XGBoost/Random Forest offer better explainability than deep neural networks
- Blockchain audit trail ensures all decisions are recorded and traceable

**5. Social Responsibility**
- ML systems should complement human judgment, not replace it entirely
- High-risk predictions should trigger manual review, not automatic rejection
- The goal is financial inclusion — not exclusion
- Models should be regularly retrained to adapt to changing economic conditions

In [ ]:
# Stop Spark session
spark.stop()
print('SparkSession stopped.')
print('\nBDA Analysis complete!')

---
## Summary

This notebook demonstrated the Big Data Analytics components required for the MicroFinML research chapter:

1. **5Vs Framework** — How Volume, Velocity, Variety, Veracity, and Value apply to microfinance data
2. **Spark Distributed Processing** — Data ingestion, transformation, and feature engineering at scale
3. **Spark MLlib Models** — Distributed training of Logistic Regression, Random Forest, and GBT
4. **Scalability Analysis** — Benchmarking local vs distributed frameworks across data sizes
5. **Blockchain Audit Trail** — Immutable, tamper-proof record of credit decisions
6. **Social Impact** — Quantified economic benefits of ML-based credit scoring
7. **Ethical Implications** — Privacy, bias, transparency, and regulatory considerations

These components elevate the project from a simple ML notebook to a **Springer-level Big Data Analytics research implementation**.